In [6]:
import pandas as pd
import pm4py

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [8]:
import wittgenstein as lw

In [9]:
def load_training_data(rules_dir):
    dataset=pd.read_csv(rules_dir,index_col=0)
    dataset=dataset.set_index("case:concept:name")

    X=dataset.drop(columns=["Class"])

    y=dataset['Class']
    print("No. of features:"+str(len(X.columns)))

    le = LabelEncoder()
    y_transformed = le.fit_transform(y)
    le_name_mapping = pd.Series(dict(zip(le.classes_,le.transform(le.classes_))))
    X_train_and_validation, X_test, y_train_and_validation, y_test = train_test_split(X,
                                                                                      y_transformed,
                                                                                      test_size=0.2,
                                                                                      stratify=y_transformed,
                                                                                      shuffle=True,
                                                                                      random_state=0)
    
    X_train, X_validation, y_train, y_validation = train_test_split(X_train_and_validation,
                                                                y_train_and_validation,
                                                                test_size=0.2,
                                                                stratify=y_train_and_validation,
                                                                shuffle=True,
                                                                random_state=0)
    return X_train, y_train, le_name_mapping

In [12]:
X_train, y_train, le_name_mapping=load_training_data("./Data/sepsis/mined_sepsis_confidences_SIRS2OrMore.csv")
X_train_replaced=X_train.fillna(-100).copy()

No. of features:4704


In [10]:
def train_ripper(X_train_log, y_train_log, pos_class_log=None, max_rule_conds_desired=None):
    #create an instance of RIPPER algorithm setting the random seed for reproducibility
    
    if type(max_rule_conds_desired)==type(None):
        ripper_clf = lw.RIPPER(random_state=0)
    else:
        ripper_clf = lw.RIPPER(random_state=0, max_rule_conds=max_rule_conds_desired)


    #train RIPPER algorithm using the training set, and passing the labels of each class
    if type(pos_class_log)==type(None):
        ripper_clf.fit(trainset=X_train_log, y=y_train_log)
    else: 
        ripper_clf.fit(trainset=X_train_log,
                       pos_class=pos_class_log,
                       y=y_train_log)
    
    return ripper_clf 

In [14]:
ripper_clf_sepsis=train_ripper(X_train_log=X_train_replaced,y_train_log=y_train)

In [37]:
ripper_clf_sepsis_false=train_ripper(X_train_log=X_train_replaced,y_train_log=y_train, pos_class_log=0)

In [39]:
ripper_clf_sepsis_false.out_model()

[["Absence(IVAntibiotics)=100.0" ^ "NotPrecedence(AdmissionNC ^ CRP)=<14.29"] V
["Absence(IVAntibiotics)=100.0"] V
["ChainResponse(Leucocytes ^ ReleaseC)=33.333333333"]]


In [ ]:
ripper_clf_sepsis=train_ripper(X_train_log=X_train_replaced,y_train_log=y_train)

In [15]:
#prints the rule set learned to differentiate the classes, V represents 'or'; ^ represents 'and'."""
ripper_clf_sepsis.out_model()

[["Absence(IVAntibiotics)=0.0"]]


In [33]:
le_name_mapping

SIRS-False    0
SIRS-True     1
dtype: int32

In [16]:
ripper_clf_sepsis.ruleset_.out_pretty()

[["Absence(IVAntibiotics)=0.0"]]


In [17]:
dir(ripper_clf_sepsis)

['VALID_HYPERPARAMETERS',
 '__abstractmethods__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_cover_remaining_positives',
 '_cover_remaining_positives_cn',
 '_ensure_has_bin_transformer',
 '_estimator_type',
 '_grow_ruleset',
 '_grow_ruleset_cn',
 '_optimize_ruleset',
 '_optimize_ruleset_cn',
 '_ruleset_frommodel',
 '_set_deprecated_fit_params',
 '_set_theory_dl_lookup',
 'add_rule',
 'algorithm_name',
 'alpha',
 'bin_transformer_',
 'class_feat',
 'classes_',
 'copy',
 'dl_allowance',
 'fit',
 'get_params',
 'init_ruleset',
 'insert_rule',
 'insert_rule_at',
 'k',
 'max_rule_conds',
 'max_rules',
 'max_total_conds',
 'n_discretize_bin

In [26]:
#show the features of the trainset, and we can see that the model correctly identifies them
ripper_clf_sepsis.trainset_features_

["'Absence(Admission IC)'",
 "'Absence(Admission NC)'",
 "'Absence(CRP)'",
 "'Absence(ER Registration)'",
 "'Absence(ER Sepsis Triage)'",
 "'Absence(ER Triage)'",
 "'Absence(IV Antibiotics)'",
 "'Absence(IV Liquid)'",
 "'Absence(LacticAcid)'",
 "'Absence(Leucocytes)'",
 "'Absence(Release A)'",
 "'Absence(Release B)'",
 "'Absence(Release C)'",
 "'Absence(Release D)'",
 "'Absence(Release E)'",
 "'Absence(Return ER)'",
 "'AlternatePrecedence(Admission IC, Admission NC)'",
 "'AlternatePrecedence(Admission IC, CRP)'",
 "'AlternatePrecedence(Admission IC, ER Registration)'",
 "'AlternatePrecedence(Admission IC, ER Sepsis Triage)'",
 "'AlternatePrecedence(Admission IC, ER Triage)'",
 "'AlternatePrecedence(Admission IC, IV Antibiotics)'",
 "'AlternatePrecedence(Admission IC, IV Liquid)'",
 "'AlternatePrecedence(Admission IC, LacticAcid)'",
 "'AlternatePrecedence(Admission IC, Leucocytes)'",
 "'AlternatePrecedence(Admission IC, Release A)'",
 "'AlternatePrecedence(Admission IC, Release B)'",
 "

In [30]:
ripper_clf_sepsis.get_params()

{'max_rules': None,
 'n_discretize_bins': 10,
 'k': 2,
 'max_rule_conds': None,
 'alpha': 1.0,
 'dl_allowance': 64,
 'max_total_conds': None,
 'random_state': 0,
 'verbosity': 0,
 'prune_size': 0.33}

In [31]:
ripper_clf_sepsis._estimator_type

'classifier'

In [32]:
ripper_clf_sepsis.__str__

<bound method RIPPER.__str__ of <RIPPER(max_rules=None, n_discretize_bins=10, k=2, max_rule_conds=None, alpha=1.0, dl_allowance=64, max_total_conds=None, random_state=0, verbosity=0, prune_size=0.33) with fit ruleset>>

In [10]:
##################################################################################################################################################################

In [11]:
X_train_rtfm, y_train_rtfm, le_name_mapping_rtfm=load_training_data("./Data/road_traffic/mined_rtfm_relabelled_confidences.csv")
X_train_rtfm_replaced=X_train_rtfm.fillna(-100).copy()

No. of features:2189


In [13]:
X_train_reduced, X_train_discard, y_train_reduced, y_discarded = train_test_split(X_train_rtfm_replaced,
                                                                                  y_train_rtfm,
                                                                                  test_size=0.8,
                                                                                  stratify=y_train_rtfm,
                                                                                  shuffle=True,
                                                                                  random_state=0)

In [9]:
ripper_clf_rtfm_unresolved=train_ripper(X_train_log=X_train_reduced,
                                        y_train_log=y_train_reduced,
                                        pos_class_log=3,
                                        max_rule_conds_desired=5)

In [10]:
ripper_clf_rtfm_unresolved.out_model()

[["AlternateResponse(SendFine ^ SendforCreditCollection)=0.0" ^ "End(SendFine)=100.0"] V
["AlternateResponse(Addpenalty ^ SendforCreditCollection)=0.0" ^ "AlternatePrecedence(Payment ^ Addpenalty)=100.0"] V
["AlternateResponse(Addpenalty ^ SendforCreditCollection)=0.0" ^ "AtLeast2(Payment)=0.0" ^ "AlternatePrecedence(ReceiveResultAppealfromPrefecture ^ InsertDateAppealtoPrefecture)=0.0" ^ "CoExistence(ReceiveResultAppealfromPrefecture ^ SendAppealtoPrefecture)=100.0" ^ "End(AppealtoJudge)=0.0"]]


In [9]:
le_name_mapping_rtfm

collected     0
dismissed     1
fully_paid    2
unresolved    3
dtype: int32

In [10]:
ripper_clf_rtfm_collected=train_ripper(X_train_log=X_train_reduced,
                                        y_train_log=y_train_reduced,
                                        pos_class_log=0,
                                        max_rule_conds_desired=5)

In [11]:
ripper_clf_rtfm_collected.out_model()

[["Absence(SendforCreditCollection)=0.0"]]


In [12]:
ripper_clf_rtfm_dismissed=train_ripper(X_train_log=X_train_reduced,
                                        y_train_log=y_train_reduced,
                                        pos_class_log=1,
                                        max_rule_conds_desired=5)

In [13]:
ripper_clf_rtfm_dismissed.out_model()

[["CoExistence(Payment ^ InsertDateAppealtoPrefecture)=0.0" ^ "Absence(ReceiveResultAppealfromPrefecture)=100.0"] V
["End(AppealtoJudge)=100.0"]]


In [14]:
ripper_clf_rtfm_fully_paid=train_ripper(X_train_log=X_train_reduced,
                                        y_train_log=y_train_reduced,
                                        pos_class_log=2,
                                        max_rule_conds_desired=5)

In [15]:
ripper_clf_rtfm_fully_paid.out_model()

[["AlternateResponse(Payment ^ Addpenalty)=0.0" ^ "Absence(SendFine)=100.0" ^ "AlternatePrecedence(CreateFine ^ Payment)=50.0"] V
["AlternateResponse(Payment ^ Addpenalty)=0.0" ^ "Absence(SendFine)=100.0"] V
["AlternateResponse(Payment ^ Addpenalty)=0.0" ^ "End(Payment)=100.0" ^ "AlternatePrecedence(CreateFine ^ Payment)=50.0" ^ "Absence(AppealtoJudge)=100.0" ^ "Absence(Addpenalty)=100.0"] V
["AlternateResponse(Payment ^ Addpenalty)=0.0" ^ "End(Payment)=100.0" ^ "AlternatePrecedence(Addpenalty ^ Payment)=50.0"] V
["AlternateResponse(Payment ^ Addpenalty)=0.0" ^ "End(Payment)=100.0" ^ "Absence(Addpenalty)=100.0"] V
["NotPrecedence(Addpenalty ^ Payment)=0.0" ^ "Absence(SendforCreditCollection)=100.0" ^ "ChainPrecedence(InsertFineNotification ^ Addpenalty)=100.0"] V
["AlternateResponse(Payment ^ SendforCreditCollection)=0.0" ^ "AlternatePrecedence(Payment ^ Addpenalty)=0.0" ^ "AlternateResponse(Addpenalty ^ ReceiveResultAppealfromPrefecture)=0.0" ^ "AlternateResponse(SendAppealtoPrefectur